In [1]:
import os
import duckdb
import pandas as pd
import sys
from pathlib import Path

repository = Path("..").resolve()
sys.path.insert(0, str(repository / "src"))

from liljegren_analysis import CampaignResults

In [2]:
runs = [
    repository / ".hsb" / "campaigns" / "era5-2025-all-six-m2-w20-v8",
    repository / ".hsb" / "runs" / "era5-2025-meteor-m4-w18" / "wbgt-comparison-20260810-231240"
]

In [ ]:
# Close the previous connection when rerunning this initialization cell.
if "con" in globals():
    try:
        con.close()
    except Exception:
        pass

con = duckdb.connect()

relations = [
    CampaignResults.from_campaign(
        run,
        connection=con,
    ).scan(
        date="2025-01-01"
    )
    for run in runs
]

if not relations:
    raise ValueError("runs is empty")

all_results = relations[0]
for relation in relations[1:]:
    all_results = all_results.union(relation)

all_results

┌─────────────┬─────────┬────────────────┬───────────────┬─────────────────────────────────────────┬─────────────┬─────────┬─────────────────────┐
│ result_date │ run_id  │ implementation │ cycle_ordinal │                 row_id                  │ row_ordinal │ status  │       value_c       │
│    date     │ varchar │    varchar     │    uint32     │                 varchar                 │   uint32    │ varchar │       double        │
├─────────────┼─────────┼────────────────┼───────────────┼─────────────────────────────────────────┼─────────────┼─────────┼─────────────────────┤
│ 2025-01-01  │ 0001    │ heatstress-jl  │             1 │ era5-global-n835000-w0360000-2025010100 │           1 │ ok      │  -31.12509484980312 │
│ 2025-01-01  │ 0001    │ heatstress-jl  │             1 │ era5-global-n835000-w0360000-2025010101 │           2 │ ok      │ -31.614817121571328 │
│ 2025-01-01  │ 0001    │ heatstress-jl  │             1 │ era5-global-n835000-w0360000-2025010102 │           3 │ ok 

In [4]:
%%time

daily_pairwise = all_results.query(
    "results",
    """
    WITH valid_results AS (
        SELECT
            result_date,
            run_id,
            implementation,
            cycle_ordinal,
            row_id,
            value_c
        FROM results
        WHERE status = 'ok'
          AND value_c IS NOT NULL
    ),
    paired_values AS (
        SELECT
            left_result.result_date,
            left_result.implementation AS implementation_a,
            right_result.implementation AS implementation_b,
            left_result.value_c - right_result.value_c AS difference_c
        FROM valid_results AS left_result
        INNER JOIN valid_results AS right_result
            ON left_result.result_date = right_result.result_date
           AND left_result.run_id = right_result.run_id
           AND left_result.cycle_ordinal = right_result.cycle_ordinal
           AND left_result.row_id = right_result.row_id
           AND left_result.implementation < right_result.implementation
    ),
    daily_statistics AS (
        SELECT
            result_date,
            implementation_a,
            implementation_b,

            count(*) AS n,
            avg(difference_c) AS mean_difference_c,
            var_samp(difference_c) AS variance_difference,

            sum(abs(difference_c)) AS absolute_difference_sum_c,
            sum(difference_c * difference_c) AS squared_difference_sum_c,

            min(difference_c) AS min_difference_c,
            max(difference_c) AS max_difference_c,
            max(abs(difference_c)) AS max_absolute_difference_c,

            approx_quantile(
                difference_c,
                [0.05, 0.25, 0.50, 0.75, 0.95]
            ) AS difference_quantiles_c,

            approx_quantile(
                abs(difference_c),
                [0.50, 0.95]
            ) AS absolute_difference_quantiles_c
        FROM paired_values
        GROUP BY
            result_date,
            implementation_a,
            implementation_b
    )
    SELECT
        result_date,
        implementation_a,
        implementation_b,

        n,
        mean_difference_c,
        variance_difference,
        sqrt(variance_difference) AS stddev_difference_c,

        absolute_difference_sum_c,
        absolute_difference_sum_c / n
            AS mean_absolute_difference_c,

        squared_difference_sum_c,
        sqrt(squared_difference_sum_c / n)
            AS rmse_c,

        min_difference_c,
        max_difference_c,
        max_absolute_difference_c,

        difference_quantiles_c[1] AS p05_difference_c,
        difference_quantiles_c[2] AS p25_difference_c,
        difference_quantiles_c[3] AS median_difference_c,
        difference_quantiles_c[4] AS p75_difference_c,
        difference_quantiles_c[5] AS p95_difference_c,

        absolute_difference_quantiles_c[1]
            AS median_absolute_difference_c,
        absolute_difference_quantiles_c[2]
            AS p95_absolute_difference_c

    FROM daily_statistics
    ORDER BY
        result_date,
        implementation_a,
        implementation_b
    """,
).df()

CPU times: total: 1min 54s
Wall time: 32.7 s


In [5]:
daily_pairwise

,result_date,implementation_a,implementation_b,n,mean_difference_c,variance_difference,stddev_difference_c,absolute_difference_sum_c,mean_absolute_difference_c,squared_difference_sum_c,...,min_difference_c,max_difference_c,max_absolute_difference_c,p05_difference_c,p25_difference_c,median_difference_c,p75_difference_c,p95_difference_c,median_absolute_difference_c,p95_absolute_difference_c
0,2025-01-01,heatstress,heatstress-jl,8440450,-0.216350,3.034228,1.741904,9.015846e+06,1.068171,2.600532e+07,...,-24.890606,13.592688,24.890606,-3.011118,-0.842091,0.079607,0.296305,2.549006,0.423010,3.682303
1,2025-01-01,heatstress,heatstress-r,8440450,-0.197452,2.722124,1.649886,7.880903e+06,0.933706,2.330502e+07,...,-60.904035,12.823499,60.904035,-2.711384,-0.697539,0.011037,0.117281,2.574384,0.256054,3.507269
2,2025-01-01,heatstress,liljegren-c,8250619,-0.186703,2.999098,1.731790,8.954110e+06,1.085265,2.503201e+07,...,-24.106961,12.726425,24.106961,-2.985021,-0.856804,0.072002,0.407478,2.584824,0.532618,3.626862
3,2025-01-01,heatstress,meteor,8440450,0.275804,1.820734,1.349346,5.775390e+06,0.684251,1.600986e+07,...,-6.599863,14.358975,14.358975,-1.397359,-0.012547,0.032870,0.209747,3.003403,0.092584,3.178924
4,2025-01-01,heatstress,pywbgt,8249720,-0.217110,2.694014,1.641345,8.247506e+06,0.999732,2.261372e+07,...,-9.378309,13.023096,13.023096,-2.921012,-0.810352,0.052869,0.175267,2.622293,0.301560,3.575323
5,2025-01-01,heatstress,thermofeel,8440450,-0.435530,3.554735,1.885400,1.011027e+07,1.197835,3.160460e+07,...,-41.968944,13.260042,41.968944,-3.642990,-1.224428,0.083911,0.253093,2.404930,0.420084,4.105221
6,2025-01-01,heatstress-jl,heatstress-r,8440450,0.018899,0.097616,0.312436,1.447458e+06,0.171491,8.269376e+05,...,-67.706797,4.701116,67.706797,-0.340803,-0.092132,-0.017887,0.164960,0.382644,0.113574,0.425157
7,2025-01-01,heatstress-jl,liljegren-c,8250619,-0.014573,0.056609,0.237927,1.146669e+06,0.138980,4.688142e+05,...,-14.196178,4.780278,14.196178,-0.265811,-0.118624,-0.019804,0.052364,0.327399,0.103357,0.378378
8,2025-01-01,heatstress-jl,meteor,8440450,0.492154,0.783592,0.885207,5.808290e+06,0.688149,8.658273e+06,...,-6.736703,18.599603,18.599603,-0.464062,-0.115445,0.041892,1.112210,2.075200,0.407384,2.077140
9,2025-01-01,heatstress-jl,pywbgt,8249720,-0.043207,0.185644,0.430864,1.447225e+06,0.175427,1.546910e+06,...,-7.951820,17.035435,17.035435,-0.358319,-0.137702,-0.053262,-0.009224,0.314328,0.097734,0.581292
